# 05_sentimiento_y_lexicon_odio_formal

## Objetivo
Aplicar el lexicón procesado al corpus formal `media_anchored`, construir indicadores
exploratorios y evaluar su comportamiento frente al nivel manual de cuatro categorías
cuando el etiquetado esté disponible.

## Entradas
- `data/processed/x_media_anchored_interactions_corpus_formal_eda.csv`
- `lexicons/processed/hatecr_lexicon.csv`
- `reports/formal_eda/manual_review_sample.csv`, en modo de solo lectura
- opcional: `lexicons/processed/sentiment_lexicon.csv`

## Salidas
- `data/processed/x_media_anchored_interactions_corpus_formal_lexicon_scored.csv`
- tablas en `outputs/tables/formal_sentiment_lexicon/`
- figuras en `outputs/figures/formal_sentiment_lexicon/`

Este notebook nunca modifica la muestra de etiquetado manual.


## 1. Setup y parámetros

In [ ]:
import importlib
import json
import os
import re
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


def find_project_root(start):
    for candidate in [start] + list(start.parents):
        if (candidate / "config").exists() and (candidate / "src").exists():
            return candidate
        child = candidate / "HateCR"
        if (child / "config").exists() and (child / "src").exists():
            return child
    raise FileNotFoundError("No se encontró la raíz del proyecto HateCR")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import src.eda as eda
import src.labels as label_utils
importlib.reload(eda)
importlib.reload(label_utils)

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
LEXICON_PROCESSED = PROJECT_ROOT / "lexicons" / "processed"
FORMAL_EDA_REPORTS = PROJECT_ROOT / "reports" / "formal_eda"
OUTPUT_TABLES = PROJECT_ROOT / "outputs" / "tables" / "formal_sentiment_lexicon"
OUTPUT_FIGURES = PROJECT_ROOT / "outputs" / "figures" / "formal_sentiment_lexicon"
OUTPUT_TABLES.mkdir(parents=True, exist_ok=True)
OUTPUT_FIGURES.mkdir(parents=True, exist_ok=True)

CORPUS_PATH = DATA_PROCESSED / "x_media_anchored_interactions_corpus_formal_eda.csv"
LEXICON_PATH = LEXICON_PROCESSED / "hatecr_lexicon.csv"
MANUAL_SAMPLE_PATH = FORMAL_EDA_REPORTS / "manual_review_sample.csv"
SENTIMENT_RESOURCE_PATH = LEXICON_PROCESSED / "sentiment_lexicon.csv"
OUTPUT_CORPUS_PATH = DATA_PROCESSED / "x_media_anchored_interactions_corpus_formal_lexicon_scored.csv"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("MANUAL_SAMPLE_PATH (solo lectura):", MANUAL_SAMPLE_PATH)
print("API y descargas automáticas: desactivadas")

## 2. Carga segura de corpus, lexicón y etiquetas

In [ ]:
id_dtypes = {
    "tweet_id": "string",
    "reply_id": "string",
    "source_post_id": "string",
    "reply_author_id_hash": "string",
}
corpus_df = eda.safe_read_csv(CORPUS_PATH, "corpus formal EDA", dtype=id_dtypes)
lexicon_df = eda.safe_read_csv(LEXICON_PATH, "lexicón HateCR", dtype=str)
manual_sample_df = eda.safe_read_csv(
    MANUAL_SAMPLE_PATH,
    "muestra manual formal (solo lectura)",
    dtype={"tweet_id": "string"},
)

if corpus_df.empty:
    raise ValueError("Falta el corpus formal EDA. Ejecuta primero los notebooks 03 y 04.")
if lexicon_df.empty:
    raise ValueError("Falta hatecr_lexicon.csv. Ejecuta primero el notebook 05a.")

print("Corpus:", len(corpus_df))
print("Lexicón:", len(lexicon_df))
print("Muestra manual disponible:", len(manual_sample_df))

## 3. Validaciones del corpus formal

In [ ]:
required_corpus = [
    "tweet_id", "text", "text_norm_no_accents", "event_id", "event_name",
    "anchor_media_handle", "source_type", "reply_author_id_hash",
]
missing = [column for column in required_corpus if column not in corpus_df.columns]
assert not missing, f"Faltan columnas: {missing}"
assert corpus_df["tweet_id"].is_unique
assert corpus_df["source_type"].eq("reply_to_media_post").all()
assert corpus_df["anchor_media_handle"].notna().all()
assert corpus_df["reply_author_id_hash"].str.fullmatch(r"[0-9a-f]{64}").all()
assert not {"author_id", "username", "screen_name"}.intersection(corpus_df.columns)

required_lexicon = {
    "term", "lemma", "category", "target_type", "severity",
    "context_dependency", "source", "include_in_classification",
}
assert required_lexicon.issubset(lexicon_df.columns)
print("[OK] Corpus y lexicón validados")

## 4. Aplicación del lexicón

Si el notebook 04 ya generó coincidencias, se reutilizan para garantizar consistencia. En caso contrario se reconstruyen mediante `src.eda.apply_lexicon_matches`.

In [ ]:
required_match_columns = {
    "lexicon_hit_count", "categorized_lexicon_hit_count",
    "uncategorized_lexicon_hit_count", "lexicon_terms_found",
    "lexicon_categories_found", "has_lexicon_match",
    "has_categorized_lexicon_match",
}

if required_match_columns.issubset(corpus_df.columns):
    working_df = corpus_df.copy()
    _, lexicon_metadata_df = eda.prepare_lexicon_index(lexicon_df)
    match_strategy = "reuse_formal_eda_matches"
else:
    working_df, lexicon_metadata_df = eda.apply_lexicon_matches(
        corpus_df,
        lexicon_df,
        text_col="text_norm_no_accents",
    )
    match_strategy = "recomputed_from_processed_lexicon"

for column in [
    "lexicon_hit_count", "categorized_lexicon_hit_count",
    "uncategorized_lexicon_hit_count",
]:
    working_df[column] = pd.to_numeric(
        working_df[column], errors="coerce"
    ).fillna(0).astype(int)

print("Estrategia:", match_strategy)
print("Filas con cualquier coincidencia:", int(working_df["has_lexicon_match"].sum()))
print("Filas con coincidencia categorizada:", int(working_df["has_categorized_lexicon_match"].sum()))

## 5. Variables binarias por categoría y `hostility_score`

In [ ]:
def safe_category_name(value):
    normalized = re.sub(r"[^a-z0-9]+", "_", str(value).strip().lower())
    return normalized.strip("_") or "unknown"


category_sets = working_df["lexicon_categories_found"].fillna("").map(
    lambda value: {item for item in str(value).split("|") if item}
)
all_categories = sorted(set().union(*category_sets.tolist()) if len(category_sets) else set())
category_columns = []
for category in all_categories:
    column = f"lexcat_{safe_category_name(category)}"
    working_df[column] = category_sets.map(lambda values: int(category in values))
    category_columns.append(column)

working_df["hostility_score"] = working_df["categorized_lexicon_hit_count"].astype(float)
working_df["hostility_score_log1p"] = np.log1p(working_df["hostility_score"])
working_df["hostility_any"] = (working_df["hostility_score"] > 0).astype(int)
working_df["hostility_category_count"] = (
    working_df[category_columns].sum(axis=1) if category_columns else 0
)
working_df["hostility_score_method"] = "count_unique_categorized_lexicon_terms"

print("Categorías binarias:", len(category_columns))
display(working_df[[
    "hostility_score", "hostility_score_log1p", "hostility_any",
    "hostility_category_count",
]].describe())

## 6. Sentimiento y emociones, si hay recursos locales

No se descargan modelos ni lexicones. Si `sentiment_lexicon.csv` no existe o no cumple el esquema mínimo, el notebook registra que el recurso no está disponible y continúa.

In [ ]:
working_df["sentiment_resource_available"] = False
working_df["sentiment_score"] = pd.NA
sentiment_status_df = pd.DataFrame([
    {"resource": str(SENTIMENT_RESOURCE_PATH), "available": False, "status": "not_found"}
])

if SENTIMENT_RESOURCE_PATH.exists():
    sentiment_df = eda.safe_read_csv(SENTIMENT_RESOURCE_PATH, "sentiment_lexicon", dtype=str)
    term_column = next(
        (column for column in ["term", "word", "lemma"] if column in sentiment_df.columns),
        None,
    )
    score_column = next(
        (column for column in ["score", "sentiment_score", "polarity"] if column in sentiment_df.columns),
        None,
    )
    if term_column and score_column:
        sentiment_df["match_term"] = sentiment_df[term_column].map(eda.normalize_for_matching)
        sentiment_df["score_numeric"] = pd.to_numeric(sentiment_df[score_column], errors="coerce")
        sentiment_map = (
            sentiment_df.dropna(subset=["match_term", "score_numeric"])
            .drop_duplicates("match_term")
            .set_index("match_term")["score_numeric"]
            .to_dict()
        )

        def score_sentiment(text):
            tokens = eda.tokenize_for_eda(text, stopwords=set(), min_length=1)
            scores = [sentiment_map[token] for token in tokens if token in sentiment_map]
            return float(np.mean(scores)) if scores else np.nan

        working_df["sentiment_score"] = working_df["text_norm_no_accents"].map(score_sentiment)
        working_df["sentiment_resource_available"] = True
        sentiment_status_df = pd.DataFrame([{
            "resource": str(SENTIMENT_RESOURCE_PATH),
            "available": True,
            "status": "applied",
            "mapped_terms": len(sentiment_map),
        }])
    else:
        sentiment_status_df = pd.DataFrame([{
            "resource": str(SENTIMENT_RESOURCE_PATH),
            "available": False,
            "status": "unsupported_schema",
        }])

display(sentiment_status_df)

## 7. Incorporación futura del etiquetado manual

La muestra se lee, valida y combina por `tweet_id`, pero nunca se modifica. Una fila
queda lista cuando contiene un nivel `0–3`, `manual_hostility` binaria y
`manual_hate_speech` binaria, todos válidos y coherentes. Hostilidad y odio se evalúan
como targets humanos separados.


In [ ]:
annotation_diagnostics = None
manual_ready_df = pd.DataFrame()
lexicon_vs_manual_df = pd.DataFrame()
lexicon_manual_metrics_df = pd.DataFrame()

if not manual_sample_df.empty:
    prepared_labels_df, annotation_diagnostics = label_utils.prepare_manual_annotations(
        manual_sample_df,
        strict=False,
    )
    manual_ready_df = prepared_labels_df[
        prepared_labels_df["annotation_ready_for_training"]
    ].copy()

    for name, table in annotation_diagnostics.items():
        eda.atomic_to_csv(table, OUTPUT_TABLES / f"annotation_{name}.csv")

    if not manual_ready_df.empty:
        label_columns = [
            "tweet_id", "hostility_relevance", "manual_hostility",
            "manual_hate_speech", "notes",
            "hostility_relevance_normalized", "manual_hostility_normalized",
            "manual_hate_speech_normalized", "hostility_level_label",
            "y_hostility", "y_severe_or_identity", "y_hate_speech",
            "y_hostility_multiclass", "annotation_valid",
        ]
        lexicon_vs_manual_df = manual_ready_df[label_columns].merge(
            working_df[[
                "tweet_id", "hostility_any", "hostility_score",
                "categorized_lexicon_hit_count", "event_id",
                "anchor_media_handle",
            ]],
            on="tweet_id",
            how="left",
        )

        def binary_metrics(y_true, y_pred, target_name):
            truth = pd.to_numeric(y_true, errors="coerce")
            pred = pd.to_numeric(y_pred, errors="coerce")
            valid = truth.isin([0, 1]) & pred.isin([0, 1])
            truth = truth[valid].astype(int)
            pred = pred[valid].astype(int)
            tp = int(((truth == 1) & (pred == 1)).sum())
            tn = int(((truth == 0) & (pred == 0)).sum())
            fp = int(((truth == 0) & (pred == 1)).sum())
            fn = int(((truth == 1) & (pred == 0)).sum())
            precision = tp / (tp + fp) if (tp + fp) else 0.0
            recall = tp / (tp + fn) if (tp + fn) else 0.0
            f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
            accuracy = (tp + tn) / len(truth) if len(truth) else 0.0
            return {
                "target": target_name,
                "n": len(truth),
                "tp": tp, "tn": tn, "fp": fp, "fn": fn,
                "accuracy": accuracy,
                "precision": precision,
                "recall": recall,
                "f1": f1,
            }

        lexicon_manual_metrics_df = pd.DataFrame([
            binary_metrics(
                lexicon_vs_manual_df["y_hostility"],
                lexicon_vs_manual_df["hostility_any"],
                "manual_hostility_binary",
            ),
            binary_metrics(
                lexicon_vs_manual_df["y_severe_or_identity"],
                lexicon_vs_manual_df["hostility_any"],
                "level_2_or_3",
            ),
            binary_metrics(
                lexicon_vs_manual_df["y_hate_speech"],
                lexicon_vs_manual_df["hostility_any"],
                "manual_hate_speech_binary",
            ),
        ])
        eda.atomic_to_csv(
            lexicon_vs_manual_df,
            OUTPUT_TABLES / "lexicon_vs_manual_rows.csv",
        )
        eda.atomic_to_csv(
            lexicon_manual_metrics_df,
            OUTPUT_TABLES / "lexicon_vs_manual_metrics.csv",
        )

print("Niveles manuales completos y válidos:", len(manual_ready_df))
if not lexicon_manual_metrics_df.empty:
    display(lexicon_manual_metrics_df)


## 8. Tablas por evento, medio y categoría

In [ ]:
def aggregate_hostility(frame, group_columns):
    return (
        frame.groupby(group_columns, dropna=False)
        .agg(
            total_rows=("tweet_id", "size"),
            hostility_any_sum=("hostility_any", "sum"),
            hostility_score_mean=("hostility_score", "mean"),
            hostility_score_median=("hostility_score", "median"),
            unique_authors=("reply_author_id_hash", "nunique"),
        )
        .reset_index()
        .assign(
            hostility_any_pct=lambda table: (
                table["hostility_any_sum"] / table["total_rows"] * 100
            ).round(2)
        )
    )


table_by_event = aggregate_hostility(working_df, ["event_id", "event_name"])
table_by_media = aggregate_hostility(
    working_df, ["anchor_media_id", "anchor_media_handle"]
)
table_by_event_media = aggregate_hostility(
    working_df,
    ["event_id", "anchor_media_id", "anchor_media_handle"],
)
table_by_source_type = aggregate_hostility(working_df, ["source_type"])

category_long_df = working_df[[
    "tweet_id", "event_id", "anchor_media_handle", "lexicon_categories_found"
]].copy()
category_long_df["category"] = category_long_df["lexicon_categories_found"].fillna("").str.split("|")
category_long_df = category_long_df.explode("category")
category_long_df = category_long_df[category_long_df["category"].fillna("").ne("")]
category_summary_df = (
    category_long_df.groupby(["event_id", "category"], dropna=False)
    .agg(n_rows=("tweet_id", "nunique"))
    .reset_index()
    .sort_values("n_rows", ascending=False)
)

tables = {
    "hostility_by_event.csv": table_by_event,
    "hostility_by_media.csv": table_by_media,
    "hostility_by_event_media.csv": table_by_event_media,
    "hostility_by_source_type.csv": table_by_source_type,
    "hostility_category_long.csv": category_long_df,
    "hostility_category_summary.csv": category_summary_df,
    "sentiment_resource_status.csv": sentiment_status_df,
}
for filename, table in tables.items():
    eda.atomic_to_csv(table, OUTPUT_TABLES / filename)

display(table_by_event)
display(table_by_media.sort_values("total_rows", ascending=False))

## 9. Gráficos exploratorios

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
plot_event = table_by_event.sort_values("hostility_any_pct", ascending=False)
ax.bar(plot_event["event_name"], plot_event["hostility_any_pct"], color="#D1495B")
ax.set_title("Coincidencia léxica categorizada por evento")
ax.set_ylabel("Porcentaje de replies")
ax.tick_params(axis="x", rotation=35)
for label in ax.get_xticklabels():
    label.set_ha("right")
fig.tight_layout()
fig.savefig(OUTPUT_FIGURES / "hostility_rate_by_event.png", dpi=170)
plt.close(fig)

fig, ax = plt.subplots(figsize=(10, 6))
plot_media = table_by_media.sort_values("hostility_any_pct").tail(20)
ax.barh(plot_media["anchor_media_handle"], plot_media["hostility_any_pct"], color="#087E8B")
ax.set_title("Coincidencia léxica categorizada por medio ancla")
ax.set_xlabel("Porcentaje de replies")
fig.tight_layout()
fig.savefig(OUTPUT_FIGURES / "hostility_rate_by_media.png", dpi=170)
plt.close(fig)

if not category_summary_df.empty:
    plot_categories = (
        category_summary_df.groupby("category")["n_rows"].sum()
        .sort_values().tail(20)
    )
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(plot_categories.index, plot_categories.values, color="#12355B")
    ax.set_title("Categorías del lexicón más frecuentes")
    ax.set_xlabel("Replies únicas")
    fig.tight_layout()
    fig.savefig(OUTPUT_FIGURES / "lexicon_category_counts.png", dpi=170)
    plt.close(fig)

print("Figuras preparadas en:", OUTPUT_FIGURES)

## 10. Exportación reproducible

In [ ]:
summary_df = pd.DataFrame([
    {"metric": "corpus_rows", "value": len(working_df)},
    {"metric": "lexicon_rows", "value": len(lexicon_df)},
    {"metric": "match_strategy", "value": match_strategy},
    {"metric": "rows_with_any_lexicon_match", "value": int(working_df["has_lexicon_match"].sum())},
    {"metric": "rows_with_categorized_match", "value": int(working_df["hostility_any"].sum())},
    {"metric": "manual_sample_rows", "value": len(manual_sample_df)},
    {"metric": "manual_ready_rows", "value": len(manual_ready_df)},
    {"metric": "sentiment_resource_available", "value": bool(working_df["sentiment_resource_available"].any())},
    {"metric": "methodological_status", "value": "exploratory_not_hate_speech_classification"},
])

eda.atomic_to_csv(summary_df, OUTPUT_TABLES / "run_summary.csv")
eda.atomic_to_csv(lexicon_metadata_df, OUTPUT_TABLES / "lexicon_index_metadata.csv")
eda.atomic_to_csv(working_df, OUTPUT_CORPUS_PATH)

print("Corpus puntuado:", OUTPUT_CORPUS_PATH)
print("Tablas:", OUTPUT_TABLES)
print("Figuras:", OUTPUT_FIGURES)
display(summary_df)

## 11. Advertencia metodológica

1. `hostility_score` cuenta términos categorizados únicos y no mide por sí solo intención ni contexto.
2. La muestra manual está enriquecida por el lexicón; precisión y recall sirven para validar el filtro, pero no para estimar prevalencia en todo el corpus.
3. `hostility_relevance`, `manual_hostility` y `manual_hate_speech` son tres decisiones humanas complementarias.
4. `manual_hostility` y `manual_hate_speech` son binarias e independientes: puede existir hostilidad `1` con odio `0`.
5. `y_hostility` conserva `manual_hostility`; `y_hate_speech` conserva `manual_hate_speech`; el código no las sustituye con el nivel.
6. El nivel `3` identifica el fundamento identitario, cultural o ideológico del ataque y sirve para comprobar coherencia con odio `1`.
7. Las categorías sin clasificar y la dependencia contextual pueden producir falsos positivos.
8. El análisis de sentimiento solo se aplica si existe un recurso local documentado; no se realizan descargas automáticas.
9. Los resultados agregados no reemplazan lectura contextual ni adjudicación de casos dudosos.
